In [1]:
!python -m pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install kaggle

Defaulting to user installation because normal site-packages is not writeable


In [1]:
# %%
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import joblib
from tensorflow.keras.preprocessing.image import load_img, img_to_array


In [2]:
# Set paths
DATASET_DIR = r"c:\Users\Vibha Poojary\Downloads\archive\plantvillage dataset\color"

In [3]:
import os
print(os.listdir(r'c:\Users\Vibha Poojary\Downloads\archive\plantvillage dataset\color'))

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Sp

In [4]:
import os
print(os.path.exists(r'c:\Users\Vibha Poojary\Downloads\archive\plantvillage dataset\color'))

True


In [5]:
# Check if path exists
if not os.path.exists(DATASET_DIR):
    print(f"Warning: Dataset directory not found: {DATASET_DIR}")
    # You may want to set DATASET_DIR to 

In [6]:
import os
print(os.listdir(DATASET_DIR))

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Sp

In [7]:
def load_data(dataset_dir, img_size=(20, 20)):
    X, y = [], []
    for label in os.listdir(dataset_dir):
        label_dir = os.path.join(dataset_dir, label)
        if not os.path.isdir(label_dir):
            continue
        
        for img_file in os.listdir(label_dir):
            img_path = os.path.join(label_dir, img_file)
            try:
                img = load_img(img_path, target_size=img_size)   # CHANGED TO 20x20
                img_arr = img_to_array(img).flatten()
                X.append(img_arr)
                y.append(label)
            except Exception as e:
                print(f"Skipping {img_path}: {e}")
    
    return np.array(X), np.array(y)


In [8]:
X, y = load_data(DATASET_DIR, img_size=(20, 20))
print("Loaded images:", len(X))
print("Loaded labels:", len(y))


Loaded images: 49179
Loaded labels: 49179


In [9]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)


In [11]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [12]:
pca = PCA(n_components=200)     # You can try 100–300
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)


In [14]:
model = SVC(kernel='rbf', C=10, gamma='scale')
model.fit(X_train_pca, y_train)

y_pred = model.predict(X_test_pca)
print("PCA + SVM Accuracy:", accuracy_score(y_test, y_pred))


PCA + SVM Accuracy: 0.8284871899145995


In [15]:
joblib.dump(model, "model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(pca, "pca.pkl")

print("All models saved successfully.")


All models saved successfully.


In [16]:
def predict_image(image_path):
    img = load_img(image_path, target_size=(20, 20))   # CHANGED TO 20x20
    img_arr = img_to_array(img).flatten().reshape(1, -1)

    # Apply scaler and PCA
    img_scaled = scaler.transform(img_arr)
    img_pca = pca.transform(img_scaled)

    pred = model.predict(img_pca)
    return le.inverse_transform(pred)[0]


In [17]:
test_img = r"c:\Users\Vibha Poojary\OneDrive\Desktop\Test_datset\WhatsApp Image 2025-11-23 at 12.03.29_2ef49183.jpg"

result = predict_image(test_img)
print("Predicted Disease:", result)


Predicted Disease: Orange___Haunglongbing_(Citrus_greening)
